In [1]:
import pandas as pd
import re
from typing import List

In [2]:
def transpose_columns(df,fixed_cols,sep_col = "."):
    '''
    Retourne le dataframe avec les colonnes non fixes pivotées

            Parameters:
                   df: dataframe source
                   fixed_cols: colonnes fixes à conserver
                   sep_col: le séparateur entre le nom de colonne et le numéro (ex: "." si la colonne est "Colonne.1")

            Returns:
                    result: le dataframe final avec les colonnes pivotées
    '''
    # Colonnes répétées
    repeated_cols = [c for c in df.columns if c not in fixed_cols]

    sep_escaped = re.escape(sep_col)
    # Récupération des noms de base
    base_names = sorted({
        re.sub(rf"{sep_escaped}\d+$", "", col)
        for col in repeated_cols
    })
    
    # Nombre de groupes
    nb_groups = max(
        int(re.search(rf"{sep_escaped}(\d+)$", c).group(1))
        if re.search(rf"{sep_escaped}(\d+)$", c)
        else 0
        for c in repeated_cols
    ) + 1
    
    dfs = []
    
    for i in range(nb_groups):
    
        cols = {}
        for base in base_names:
    
            col_name = base if i == 0 else f"{base}{sep_col}{i}"
    
            if col_name in df.columns:
                cols[col_name] = base
    
        tmp = df[fixed_cols].copy()
    
        for old_name, new_name in cols.items():
            tmp[new_name] = df[old_name]
    
        dfs.append(tmp)
    
    result = pd.concat(dfs, ignore_index=True)
    result = result.dropna(
        subset=base_names,
        how="all"
    )
    return result

In [3]:
def pct_cols(df):
    '''
    transforme les colonnes contenant "%" en float

            Parameters:
                    df: dataframe source

            Returns:
                    df: le dataframe transformé
    '''
    pct_cols = [c for c in df.columns if "%" in c]

    for col in pct_cols:
        df[col] = (
            pd.to_numeric(
                df[col]
                    .astype(str)
                    .str.replace("%", "", regex=False)
                    .str.replace(",", ".", regex=False),
                errors="coerce"
            )
        )
    return df

In [4]:
def col_to_bool(df,col,val_col):
    '''
    transforme les colonnes en booléen avec correspondance de valeurs

            Parameters:
                    df: dataframe source
                    col: la colonne à transformer
                    val_col: la valeur pour True (False correspondant toujours à vide)

            Returns:
                    df: le dataframe transformé
    '''
    df[col] = df[col].map({val_col:True ,None:False})
    return df

In [5]:
def harmonize_sex(df,col):
    '''
    harmonise les colonnes sexe en F/M

            Parameters:
                    df: dataframe source
                    col: la colonne à transformer

            Returns:
                    df: le dataframe transformé
    '''
    df[col] = df[col].map({'FEMININ':'F' ,'MASCULIN':'M'})
    return df

In [6]:
def normalize_name_columns(
    df: pd.DataFrame,
    columns_to_normalize: List[str]
) -> pd.DataFrame:
    """
    Normalize name columns in a DataFrame by applying uppercase/title case and stripping whitespace.
    
    This function applies:
    - Uppercase and strip to columns containing 'nom_' (last names)
    - Title case and strip to columns containing 'prenom_' (first names)
    
    Args:
        df: The DataFrame to normalize.
        columns_to_normalize: List of column names to normalize.
    
    Returns:
        pd.DataFrame: The DataFrame with normalized columns (modified in place).
    """
    for col in columns_to_normalize:
        if col in df.columns:
            if 'prenom_' in col:
                df[col] = df[col].str.title().str.strip()
            elif 'nom_' in col:
                df[col] = df[col].str.upper().str.strip()
    
    return df

In [7]:
def correct_prenom_spec(
    dataframes: List[pd.DataFrame],
    corrections: List[tuple[str, str, str]]
) -> None:
    """
    Corrects 'prenom_candidat' and 'prenom_remplacant' values in multiple DataFrames based on matching criteria.
    
    This function updates the 'prenom_candidat' and 'prenom_remplacant' column in-place for rows where both
    'nom_remplacant' and 'prenom_candidat' and 'prenom_remplacant' match the specified criteria.
    
    Args:
        dataframes: List of pandas DataFrames to apply corrections to.
        corrections: List of tuples containing (last_name, old_first_name, new_first_name).
    
    Returns:
        None: Modifies DataFrames in-place.
    
    Example:
        >>> corrections = [('FLEURY', 'Michel', 'Michel Guillaume')]
        >>> correct_prenom_spec([df1, df2], corrections)
    """
    for df in dataframes:
        for nom, old_prenom, new_prenom in corrections:
            mask = (df['nom_candidat'] == nom) & (df['prenom_candidat'] == old_prenom)
            df.loc[mask, 'prenom_candidat'] = new_prenom
            mask = (df['nom_remplacant'] == nom) & (df['prenom_remplacant'] == old_prenom)
            df.loc[mask, 'prenom_remplacant'] = new_prenom

In [8]:
def apply_prenom_cleaning(
    dataframes: Dict[str, pd.DataFrame],
    columns: List[str],
    cleaning_function: callable
) -> None:
    """
    Apply a cleaning function to specified columns across multiple DataFrames in-place.
    
    This function iterates through a dictionary of DataFrames and applies a cleaning
    function to specified columns, falling back to original values if cleaning fails.
    
    Args:
        dataframes: Dictionary mapping DataFrame names to DataFrame objects.
        columns: List of column names to clean in each DataFrame.
        cleaning_function: Function to apply for cleaning the column values.
    
    Returns:
        None: Modifies DataFrames in-place.
    
    Example:
        >>> dfs = {'df1': pd.DataFrame({'prenom': ['JEAN', 'MARIE']})}
        >>> apply_prenom_cleaning(dfs, ['prenom'], str.lower)
    """
    for df_name, df in dataframes.items():
        for column in columns:
            if column in df.columns:
                df[column] = (
                    df[column]
                    .map(cleaning_function, na_action='ignore')
                    .fillna(df[column])
                )

In [9]:
def add_new_candidates_id(df_id,new_df, cols=['nom_candidat', 'prenom_candidat', 'date_naissance_candidat']):
    '''
    ajoute les nouveaux candidats au df générant les ids

            Parameters:
                    df_id: dataframe générant les ids
                    new_df: le dataframe qui contient les nouveaux candidats à considérer
                    col: les colonnes à prendre en compte (candidat ou remplacant)
    '''
    trios_values = new_df[cols].drop_duplicates()
    trios_values.rename(columns={'nom_remplacant':'nom_candidat', 'prenom_remplacant':'prenom_candidat', 'date_naissance_remplacant':'date_naissance_candidat'}, inplace = True)
    new_trios = trios_values.merge(
        df_id[['nom_candidat', 'prenom_candidat', 'date_naissance_candidat']], 
        on=['nom_candidat', 'prenom_candidat', 'date_naissance_candidat'],
        how='left',
        indicator=True
    ).query('_merge == "left_only"').drop(columns=['_merge'])
    if len(new_trios) > 0:
        new_trios.insert(0, 'id_candidat', range(len(df_id) + 1, len(df_id) + len(new_trios) + 1))
        df_id = pd.concat([df_id, new_trios], ignore_index=True)
    print('{0} candidats ajoutés'.format(len(new_trios)))
    return df_id

# Elections législatives 2002

## Candidatures

In [10]:
fichier_candidature_2002_t1 = "2002/sources/legislatives-2002-candidatures-france-entiere-tour-1-2024-06-21-10h50.csv"
candidature_2002_t1 = pd.read_csv(fichier_candidature_2002_t1, sep = ";")

In [11]:
fichier_candidature_2002_t2 = "2002/sources/legislatives-2002-candidatures-france-entiere-tour-2-2024-06-21-10h52.csv"
candidature_2002_t2 = pd.read_csv(fichier_candidature_2002_t2, sep = ";")

## Résultats

In [12]:
fichier_resultats_2002 = "2002/sources/96f64b379471595907b6abb0267e7b9e36a1d9ccd06521e377c8330c3130de.xls"
# Colonnes fixes
fixed_cols = [
    "Code du département", "Libellé du département", "Code de la circonscription", "Libellé de la circonscription", "Inscrits", "Abstentions", "% Abs/Ins", "Votants", "% Vot/Ins", "Blancs et nuls", "% BlNuls/Ins", "% BlNuls/Vot", "Exprimés", "% Exp/Ins", "% Exp/Vot"
]

In [13]:
resultats_2002_t1 = pd.read_excel(fichier_resultats_2002, sheet_name="Circo leg T1")
resultats_2002_t1_tr = transpose_columns(resultats_2002_t1,fixed_cols)
resultats_2002_t1_tr.to_csv("2002/work/resultats_2022_t1.csv", index=False)

In [14]:
resultats_2002_t2 = pd.read_excel(fichier_resultats_2002, sheet_name="Circo leg T2")
resultats_2002_t2_tr = transpose_columns(resultats_2002_t2,fixed_cols)
resultats_2002_t2_tr.to_csv("2002/work/resultats_2022_t2.csv", index=False)

# Elections législatives 2024

## Candidatures

In [15]:
fichier_candidature_2024_t1 = "2024/sources/legislatives-2024-candidatures-france-entiere-tour-1-2024-06-28.csv"
candidature_2024_t1 = pd.read_csv(fichier_candidature_2024_t1, sep = ";")

In [16]:
fichier_candidature_2024_t2 = "2024/sources/legislatives-2024-candidatures-france-entiere-tour-2-2024-07-05.csv"
candidature_2024_t2 = pd.read_csv(fichier_candidature_2024_t2, sep = ";")

## Résultats

In [17]:
# Colonnes fixes
fixed_cols = [
    "Code département","Libellé département","Code circonscription législative","Libellé circonscription législative","Inscrits","Votants","% Votants","Abstentions","% Abstentions","Exprimés","% Exprimés/inscrits","% Exprimés/votants","Blancs","% Blancs/inscrits","% Blancs/votants","Nuls","% Nuls/inscrits","% Nuls/votants"
]

In [18]:
fichier_2024_t1 = "2024/sources/resultats-definitifs-par-circonscriptions-legislatives.csv"
resultats_2024_t1 = pd.read_csv(fichier_2024_t1, sep = ";", decimal=",")
resultats_2024_t1 = pct_cols(resultats_2024_t1)
resultats_2024_t1_tr = transpose_columns(resultats_2024_t1,fixed_cols," ")
resultats_2024_t1_tr.to_csv("2024/work/resultats_2024_t1.csv", index=False)

In [19]:
fichier_2024_t2 = "2024/sources/resultats-definitifs-par-circonscription.csv"
resultats_2024_t2 = pd.read_csv(fichier_2024_t2, sep = ";", decimal=",",quotechar='"')
resultats_2024_t2 = pct_cols(resultats_2024_t2)
resultats_2024_t2_tr = transpose_columns(resultats_2024_t2,fixed_cols," ")
resultats_2024_t2_tr.to_csv("2024/work/resultats_2024_t2.csv", index=False)

# Préparation des données

## Renommage des colonnes

In [20]:
col_to_rename = {
    'Code département':'code_departement',
    'Code du département':'code_departement',
                     
    'Libellé du département':'lbl_departement',
    'Département':'lbl_departement',
    'Libellé département':'lbl_departement',

    'Code de la circonscription':'code_circonscription',
    'Code circonscription':'code_circonscription',
    'Code circonscription législative':'code_circonscription',

    'Libellé circonscription':'lbl_circonscription',
    'Nom_de_la_Circonscription':'lbl_circonscription',
    'Libellé de la circonscription':'lbl_circonscription',
    'Libellé circonscription législative':'lbl_circonscription',

    'Numéro de panneau':'num_panneau',
    'N° dépôt':'num_depot',

    'Nom du candidat':'nom_candidat',
    'Nom':'nom_candidat',
    'Nom candidat':'nom_candidat',

    'Prénom du candidat':'prenom_candidat',
    'Prénom':'prenom_candidat',
    'Prénom candidat':'prenom_candidat',

    'Date de naissance du candidat':'date_naissance_candidat',

    'Sexe du candidat':'sexe_candidat',
    'Sexe candidat':'sexe_candidat',
    'Sexe':'sexe_candidat',
    
    'Sortant':'bool_sortant',

    'Profession':'profession',

    'Nom remplaçant':'nom_remplacant',
    'Prénom remplaçant':'prenom_remplacant',
    'Date de naissance remplaçant':'date_naissance_remplacant',
    'Sexe remplaçant':'sexe_remplacant',
    'Sortant remplaçant':'bool_sortant_remplacant',

    'Code nuance':'code_nuance',
    'Nuance':'code_nuance',
    'Nuance candidat':'code_nuance',
    'Nuance du candidat':'lbl_nuance',

    '% Abs/Ins':'pct_abstentions_inscrits',
    '% Abstentions':'pct_abstentions',
    '% Blancs/inscrits':'pct_blancs_inscrits',
    '% Blancs/votants':'pct_blancs_votants',
    '% BlNuls/Ins':'pct_blancsnuls_inscrits',
    '% BlNuls/Vot':'pct_blancsnuls_votants',
    '% Nuls/inscrits':'pct_nuls_inscrits',
    '% Nuls/votants':'pct_nuls_votants',
    '% Exp/Ins':'pct_exprimes_inscrits',
    '% Exp/Vot':'pct_exprimes_votants',
    '% Exprimés/inscrits':'pct_exprimes_inscrits',
    '% Exprimés/votants':'pct_exprimes_votants',
    '% Voix/Exp':'pct_voix_exprimes',
    '% Voix/exprimés':'pct_voix_exprimes',
    '% Voix/Ins':'pct_voix_inscrits',
    '% Voix/inscrits':'pct_voix_inscrits',
    '% Vot/Ins':'pct_votants_inscrits',
    '% Votants':'pct_votants',

    'Abstentions':'nb_abstentions',
    'Blancs':'nb_blancs',
    'Blancs et nuls':'nb_blancsnuls',
    'Exprimés':'nb_exprimes',
    'Inscrits':'nb_inscrits',
    'Nuls':'nb_nuls',
    'Voix':'nb_voix',
    'Votants':'nb_votants',
    
    'Elu':'bool_elu'
                }

In [21]:
candidature_2002_t1.rename(columns=col_to_rename, inplace = True)
candidature_2002_t2.rename(columns=col_to_rename, inplace = True)

resultats_2002_t1_tr.rename(columns=col_to_rename, inplace = True)
resultats_2002_t2_tr.rename(columns=col_to_rename, inplace = True)

candidature_2024_t1.rename(columns=col_to_rename, inplace = True)
candidature_2024_t2.rename(columns=col_to_rename, inplace = True)

resultats_2024_t1_tr.rename(columns=col_to_rename, inplace = True)
resultats_2024_t2_tr.rename(columns=col_to_rename, inplace = True)

## Transformation des booléens

In [22]:
candidature_2002_t1 = col_to_bool(candidature_2002_t1,'bool_sortant','OUI')
candidature_2002_t1 = col_to_bool(candidature_2002_t1,'bool_sortant_remplacant','OUI')

candidature_2002_t2 = col_to_bool(candidature_2002_t2,'bool_sortant','OUI')
candidature_2002_t2 = col_to_bool(candidature_2002_t2,'bool_sortant_remplacant','OUI')

candidature_2024_t1 = col_to_bool(candidature_2024_t1,'bool_sortant','OUI')
candidature_2024_t1 = col_to_bool(candidature_2024_t1,'bool_sortant_remplacant','OUI')

candidature_2024_t2 = col_to_bool(candidature_2024_t1,'bool_sortant','OUI')
candidature_2024_t2 = col_to_bool(candidature_2024_t1,'bool_sortant_remplacant','OUI')

resultats_2024_t1_tr = col_to_bool(resultats_2024_t1_tr,'bool_elu','élu')
resultats_2024_t2_tr = col_to_bool(resultats_2024_t2_tr,'bool_elu','élu')

## Harmonisation des valeurs

### Sexe

In [23]:
resultats_2024_t1_tr = harmonize_sex(resultats_2024_t1_tr,'sexe_candidat')
resultats_2024_t2_tr = harmonize_sex(resultats_2024_t2_tr,'sexe_candidat')

### Noms et Prénoms

In [24]:
# Define the columns to normalize
name_columns = ['nom_candidat', 'prenom_candidat', 'nom_remplacant', 'prenom_remplacant']

# Apply normalization to all DataFrames
dataframes = [
    candidature_2002_t1,
    candidature_2002_t2,
    resultats_2002_t1_tr,
    resultats_2002_t2_tr,
    candidature_2024_t1,
    candidature_2024_t2,
    resultats_2024_t1_tr,
    resultats_2024_t2_tr
]

for df in dataframes:
    normalize_name_columns(df, name_columns)

In [25]:
# BOUTRIN	Louis à passer en BOUTRIN Georges Louis
# BRIVARY	Anne à passer en BRIVARY	Anne-Isabelle
# CAROLE	Francis à passer en CAROLE	Luc Francis
# FLEURY	Michel à passer en FLEURY	Michel Guillaume
# VIAULE	Serge à passer en VIAULE	Sergi
# LOMBARD	Didier, Michel à passer en LOMBARD Didier
# CASSETTE	Danielle à passer en CASSETTE	Danièle
# DEROCHE	François DEROCHE à passer en 	François-Xavier
# CAMONIN	Jean CAMONIN à passer en 	Jean-Louis
# GRAU	Lucile et GRAU	Lucila semblent des personnes différentes


# Define the corrections once
NAME_CORRECTIONS = [
    ('BOUTRIN', 'Louis', 'Georges Louis'),
    ('BRIVARY', 'Anne', 'Anne-Isabelle'),
    ('CAROLE', 'Francis', 'Luc Francis'),
    ('FLEURY', 'Michel', 'Michel Guillaume'),
    ('VIAULE', 'Serge', 'Sergi'),
    ('LOMBARD', 'Didier, Michel', 'Didier'),
    ('DEROCHE', 'François', 'François-Xavier'),
    ('CASSETTE', 'Danielle', 'Danièle'),
    ('CAMONIN', 'Jean', 'Jean-Louis'),
]

# Apply corrections to all DataFrames
correct_prenom_spec(
    [candidature_2002_t1, candidature_2002_t2, candidature_2024_t1, candidature_2024_t2],
    NAME_CORRECTIONS
)


In [26]:
# Certains candidats n'ont pas le prénom écrit pareil d'une candidature à l'autre, besoin de clean :

clean_prenoms = {    
    'Agnes':  'Agnès',
    'Ahamada':  'Ahumad',
    'Aicha':  'Aïcha',
    'Andre':  'André',
    'A.Catherine':  'Anne-Catherine',
    'A.Laure':  'Anne-Laure',
    'A.Marie':  'Anne-Marie',
    'Daniele':  'Danièle',
    'Danielle':  'Danielle',
    'Deolinda':  'Déolinda',
    'Didier, michel':  'Didier',
    'Elodie':  'Élodie',
    'Eric':  'Éric',
    'Florence Jacqueline Chantal':  'Florence',
    'Francois':  'François',
    'Francoise':  'Françoise',
    'Frederic':  'Frédéric',
    'Gael':  'Gaël',
    'Genevieve':  'Geneviève',
    'Geraldine':  'Géraldine',
    'Gerard':  'Gérard',
    'Gisele':  'Gisèle',
    'Helene':  'Hélène',
    'Herve':  'Hervé',
    'J.Marc':  'Jean-Marc',
    'Jean Marc':  'Jean-Marc',
    'J.Yves':  'Jean-Yves',
    'J.Carles':  'Jean-Carles',
    'J.Claude':  'Jean-Claude',
    'Jean Claude':  'Jean-Claude',
    'J.Felix':  'Jean-Félix',
    'J.François':  'Jean-François',
    'J.Francois':  'Jean-François',
    'Jean Jacques':  'Jean-Jacques',
    'J.Lin':  'Jean-Lin',
    'J.Louis':  'Jean-Louis',
    'J.Loup':  'Jean-Loup',
    'J. Luc':  'Jean-Luc',
    'J.Luc':  'Jean-Luc',
    'J.Marie':  'Jean-Marie',
    'J.Michel':  'Jean-Michel',
    'J.Paul':  'Jean-Paul',
    'J.Philippe':  'Jean-Philippe',
    'J.Pierre':  'Jean-Pierre',
    'Jean Yves':  'Jean-Yves',
    'Jerome':  'Jérôme',
    'Joel':  'Joël',
    'Joelle':  'Joëlle',
    'Jose':  'José',
    'Lea':  'Léa',
    'Francis':  'Luc Francis',
    'Marie-Claude':  'Marie Claude',
    'M.Claude':  'Marie Claude',
    'M.Helene':  'Marie Hélène',
    'M.Louise':  'Marie Louise',
    'M. Christine':  'Marie-Christine',
    'M.Claire':  'Marie-Claire',
    'M.France':  'Marie-France',
    'M.George':  'Marie-George',
    'M.Josée':  'Marie-José',
    'M.Josee':  'Marie-José',
    'M.Michele':  'Marie-Michèle',
    'M.Renee':  'Marie-Renée',
    'M.Thérèse':  'Marie-Thérèse',
    'Michele':  'Michèle',
    'Michèle Marie Rose Albertine':  'Michèle',
    'Noelle':  'Noëlle',
    'Regis':  'Régis',
    'Remi':  'Rémi',
    'Remy':  'Rémy',
    'Rene':  'René',
    'Renee':  'Renée',
    'Sebastien':  'Sébastien',
    'Stephane':  'Stéphane',
    'Valerie':  'Valérie',
    'Valery':  'Valéry',
    'Veronique':  'Véronique',
    'Néjima': 'Nejima',
}

# Define the DataFrames to process
dataframes_to_clean = {
    'candidature_2002_t1': candidature_2002_t1,
    'candidature_2002_t2': candidature_2002_t2,
    'resultats_2002_t1_tr': resultats_2002_t1_tr,
    'resultats_2002_t2_tr': resultats_2002_t2_tr,
    'candidature_2024_t1': candidature_2024_t1,
    'candidature_2024_t2': candidature_2024_t2,
    'resultats_2024_t1_tr': resultats_2024_t1_tr,
    'resultats_2024_t2_tr': resultats_2024_t2_tr,
}

# Apply cleaning to both prenom columns
columns_to_clean = ['prenom_candidat', 'prenom_remplacant']
apply_prenom_cleaning(dataframes_to_clean, columns_to_clean, clean_prenoms)


## Calcul de l'âge

In [27]:
dates_t1 = {2002: '2002-06-09',
            2007: '2007-06-10',
            2012: '2012-06-10',
            2017: '2017-06-11',
            2022: '2022-06-12',
            2024: '2024-06-25'
           }

candidature_2002_t1['date_naissance_candidat'] = pd.to_datetime(candidature_2002_t1['date_naissance_candidat'],format="%d/%m/%Y")
candidature_2002_t1['age_candidat'] = ((pd.to_datetime(dates_t1[2002]) - candidature_2002_t1['date_naissance_candidat']).dt.days / 365.25).astype(int)

candidature_2002_t1['date_naissance_remplacant'] = pd.to_datetime(candidature_2002_t1['date_naissance_remplacant'],format="%d/%m/%Y")
candidature_2002_t1['age_remplacant'] = ((pd.to_datetime(dates_t1[2002]) - candidature_2002_t1['date_naissance_remplacant']).dt.days / 365.25).astype(int)

candidature_2024_t1['date_naissance_candidat'] = pd.to_datetime(candidature_2024_t1['date_naissance_candidat'],format="%d/%m/%Y")
candidature_2024_t1['age_candidat'] = ((pd.to_datetime(dates_t1[2024]) - candidature_2024_t1['date_naissance_candidat']).dt.days / 365.25).astype(int)

candidature_2024_t1['date_naissance_remplacant'] = pd.to_datetime(candidature_2024_t1['date_naissance_remplacant'],format="%d/%m/%Y")
candidature_2024_t1['age_remplacant'] = ((pd.to_datetime(dates_t1[2024]) - candidature_2024_t1['date_naissance_remplacant']).dt.days / 365.25).astype(int)

In [28]:
assert (candidature_2002_t1['age_candidat'] >= 18).all() and (candidature_2002_t1['age_candidat'] <= 150).all()
assert (candidature_2002_t1['age_remplacant'] >= 18).all() and (candidature_2002_t1['age_remplacant'] <= 150).all()

assert (candidature_2024_t1['age_candidat'] >= 18).all() and (candidature_2024_t1['age_candidat'] <= 150).all()
assert (candidature_2024_t1['age_remplacant'] >= 18).all() and (candidature_2024_t1['age_remplacant'] <= 150).all()

# Tables de travail

In [30]:
'''
Init du fichier candidats_id avec les candidats de 2022
candidat_id = candidature_2002_t1[['nom_candidat', 'prenom_candidat', 'date_naissance_candidat']].drop_duplicates()
candidat_id.insert(0, 'id_candidat', range(1, len(candidat_id) + 1))
candidat_id.to_csv("work/candidats_id.csv", index=False)
'''
candidat_id = pd.read_csv("work/candidats_id.csv")
candidat_id['date_naissance_candidat'] = pd.to_datetime(candidat_id['date_naissance_candidat'])

candidat_id = add_new_candidates_id(candidat_id,candidature_2002_t1,['nom_remplacant', 'prenom_remplacant', 'date_naissance_remplacant'])
candidat_id = add_new_candidates_id(candidat_id,candidature_2024_t1)
candidat_id = add_new_candidates_id(candidat_id,candidature_2024_t1,['nom_remplacant', 'prenom_remplacant', 'date_naissance_remplacant'])

candidat_id.to_csv("work/candidats_id.csv", index=False)

8444 candidats ajoutés
3672 candidats ajoutés
3842 candidats ajoutés
